# kLa Bioreactor — Post-Processing

Analyses the `kla_bioreactor` simulation output and compares against the benchmark study
benchmark (Thomas et al., CES 237, 2021, DOI: 10.1016/j.ces.2021.116538).

**Validation targets**

| Quantity | Target | Source |
|---|---|---|
| k_La | 4.1 hr-1 | benchmark Table 2, 400 RPM, 0.4 L/min |
| Mean bubble diameter (above impeller) | ~3 mm | benchmark Fig. 21 |
| Shaft power = total dissipation | < 5 % error | benchmark Fig. 20 |
| Peak O2 transfer rate | ~0.49 mmol/s | benchmark Fig. 24 |

**Output files consumed**
- `bubble_stats.csv` columns (BubbleManager::open_stats_file):  
  `step, phys_time_s, n_bubbles, d_mean_mm, d_min_mm, d_max_mm, n_O2_total_mol, dn_O2_step_mol_per_s`
- `thermodynamics.txt` -- AMReX energy-budget (col 1 = physical time [s] already)
- `forces.txt`         -- AMReX body forces / torques
- `plt?????/`          -- AMReX plotfiles (optional, for spatial O2 map)

Set `RUN_DIR` in the Parameters cell before running.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.4})

In [ ]:
# Parameters -- adjust RUN_DIR before running
RUN_DIR = Path('.')

# Physical constants (must match kla_bioreactor.inp)
C_ref   = 100.0        # mol/m3 per LB_rho unit
S       = 0.032        # Henry solubility (dimensionless)
C_g0    = 44.6         # mol/m3  (= 1/V_m at STP)
C_sat   = S * C_g0     # = 1.427 mol/m3  (C* = S * C_g)

dt_phys = 4.77465e-5   # s/step  (omega_LB / omega_phys)
dx_phys = 1.0e-3       # m/cell

# Tank geometry
T_m     = 0.18         # m
h_fluid = 0.13         # m
V_liq   = np.pi / 4.0 * T_m**2 * h_fluid   # ~3.30e-3 m3

print(f'Liquid volume : {V_liq*1e3:.2f} L')
print(f'C*            : {C_sat:.4f} mol/m3  (S={S}, C_g0={C_g0})')

## 1  Bubble statistics overview

In [ ]:
stats_file = RUN_DIR / 'bubble_stats.csv'
if not stats_file.exists():
    raise FileNotFoundError(f'Cannot find {stats_file}.  Set RUN_DIR correctly.')
bdf = pd.read_csv(stats_file)
print('Columns:', list(bdf.columns))
print(bdf.head())

In [ ]:
# Physical time vector
# BubbleManager writes column 'phys_time_s' [s] -- use it directly.
# Fall back to step * dt_phys only if that column is absent.
if 'phys_time_s' in bdf.columns:
    t = bdf['phys_time_s'].values
elif 'phys_time' in bdf.columns:
    t = bdf['phys_time'].values
else:
    t = bdf['step'].values * dt_phys
t_hr = t / 3600.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bubble count
n_col = 'n_bubbles' if 'n_bubbles' in bdf.columns else \
        next((c for c in bdf.columns if 'n_bubble' in c.lower()), None)
if n_col:
    axes[0].plot(t, bdf[n_col])
else:
    axes[0].text(0.5, 0.5, 'n_bubbles not found', ha='center', va='center',
                 transform=axes[0].transAxes)
axes[0].set(xlabel='Physical time [s]', ylabel='Number of bubbles',
            title='Bubble count vs time')

# Mean diameter
# d_mean_mm is written as d_mean_SI * 1000 inside write_stats() -- already in mm.
# Do NOT multiply by 1e3 again (that would give micrometres).
d_col = 'd_mean_mm' if 'd_mean_mm' in bdf.columns else \
        next((c for c in bdf.columns if 'd_mean' in c.lower()), None)
if d_col:
    axes[1].plot(t, bdf[d_col], label='d_mean  [mm]')  # already mm
    axes[1].axhline(3.0, color='r', ls='--', label='target  3 mm')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'd_mean_mm not found', ha='center', va='center',
                 transform=axes[1].transAxes)
axes[1].set(xlabel='Physical time [s]', ylabel='Mean bubble diameter [mm]',
            title='Mean bubble diameter vs time')

plt.tight_layout()
plt.savefig(RUN_DIR / 'bubble_overview.png', bbox_inches='tight')
plt.show()

## 2  Bubble size distribution

`bubble_stats.csv` stores only the mean/min/max diameter per write step, not
per-bubble diameters.  This plot shows the diameter statistics band over time
rather than a true PSD histogram.  For a PSD, per-bubble diameter output would
need to be added to `write_stats()`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
if d_col:
    d_mean = bdf[d_col].values
    i_qs = int(0.8 * len(t))
    d_qs_mean = d_mean[i_qs:].mean()
    ax.plot(t, d_mean, color='steelblue', label='d_mean')
    if 'd_min_mm' in bdf.columns and 'd_max_mm' in bdf.columns:
        ax.fill_between(t, bdf['d_min_mm'], bdf['d_max_mm'],
                        alpha=0.2, color='steelblue', label='d_min/d_max band')
    ax.axhline(3.0, color='r', ls='--', label='target  3 mm')
    ax.axhline(d_qs_mean, color='b', ls=':', lw=1.5,
               label=f'Quasi-steady mean = {d_qs_mean:.2f} mm')
    ax.set(xlabel='Physical time [s]', ylabel='Bubble diameter [mm]',
           title='Bubble diameter statistics vs time')
    ax.legend()
    print(f'Quasi-steady mean diam (last 20%): {d_qs_mean:.2f} mm  (target ~3 mm)')
else:
    ax.text(0.5, 0.5, 'd_mean_mm not found', ha='center', va='center',
            transform=ax.transAxes)
plt.tight_layout()
plt.savefig(RUN_DIR / 'bubble_size_distribution.png', bbox_inches='tight')
plt.show()

## 3  O2 transfer rate

Column `dn_O2_step_mol_per_s` [mol/s] from `write_stats()`:  
$$\dot{n}_T = \sum_i k_{L,i}\, A_i\,(S\,C_{g,i} - C_{f,i})$$

**Important**: `n_O2_total_mol` is the gas-phase O2 inventory (moles currently
inside bubbles).  It decreases as bubbles dissolve and must **not** be used as a
transfer rate or cumulative dissolved-O2 quantity.

In [ ]:
rate_col = 'dn_O2_step_mol_per_s' if 'dn_O2_step_mol_per_s' in bdf.columns else \
           next((c for c in bdf.columns
                 if 'dn_o2' in c.lower() or ('mol_per_s' in c.lower() and 'dn' in c.lower())),
                None)

fig, ax = plt.subplots(figsize=(8, 5))
if rate_col:
    rate_mmol = bdf[rate_col].values * 1e3   # mol/s -> mmol/s
    ax.plot(t, rate_mmol, lw=0.8, alpha=0.5, color='steelblue', label='raw')
    if len(t) > 21:
        wl = min(51, (len(t) // 4) * 2 + 1)
        ax.plot(t, savgol_filter(rate_mmol, wl, 3), lw=2, color='steelblue',
                label='Savitzky-Golay smoothed')
    ax.set_ylabel('ndot_T  [mmol/s]')
else:
    ax.text(0.5, 0.5,
            'dn_O2_step_mol_per_s not found in bubble_stats.csv.',
            ha='center', va='center', transform=ax.transAxes)
ax.axhline(0.49, color='r', ls='--', label='literature peak ~0.49 mmol/s')
ax.set(xlabel='Physical time [s]', title='Instantaneous O2 transfer rate')
ax.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / 'O2_transfer_rate.png', bbox_inches='tight')
plt.show()

## 4  k_La fit

Well-mixed saturation model (deaerated start, C_f(0)=0, constant C_g=C_g0):  
$$C_f(t) = C^*\left[1 - e^{-k_La\,t}\right], \quad C^* = S\,C_{g,0}$$

C_f(t) is obtained by integrating the instantaneous transfer rate:  
$$C_f(t) = \frac{1}{V_{\rm liq}} \int_0^t \dot{n}_T(\tau)\,d\tau$$

The constant-C_g assumption is standard practice for the literature benchmark
over the 10 s measurement window where C_f << C*.

In [ ]:
# Build C_f(t) [mol/m3] by integrating dn_O2_step_mol_per_s over time.
#
# dn_O2_step_mol_per_s is the instantaneous rate [mol/s] -- integrate with trapz.
# np.gradient(t) gives the local time step between consecutive write times [s].
#
# DO NOT use n_O2_total_mol: it is the gas-phase O2 inventory inside bubbles
# (a decreasing quantity), not cumulative dissolved O2.
if rate_col:
    rate_mol_s = bdf[rate_col].values          # mol/s
    dt_vec     = np.gradient(t)                # s between writes
    cum_O2     = np.cumsum(rate_mol_s * dt_vec)  # mol dissolved in liquid
    C_f        = cum_O2 / V_liq                # mol/m3
    print(f'Final C_f  = {C_f[-1]:.4f} mol/m3')
    print(f'C*         = {C_sat:.4f} mol/m3')
    print(f'DOT        = {C_f[-1]/C_sat*100:.1f} %')
else:
    print('Cannot compute C_f: dn_O2_step_mol_per_s column not found.')
    C_f = None

In [ ]:
TARGET_KLA_HR = 4.1   # hr-1

def saturation_model(t_s, kla_s):
    """C_f(t) = C* (1 - exp(-kLa * t))  [mol/m3]"""
    return C_sat * (1.0 - np.exp(-kla_s * t_s))

if C_f is not None and len(t) > 10 and C_f[-1] > 0:
    mask = t >= t[max(1, int(0.05 * len(t)))]
    popt, pcov = curve_fit(saturation_model, t[mask], C_f[mask],
                           p0=[TARGET_KLA_HR / 3600.0], bounds=(0, np.inf))
    kla_s  = popt[0]
    kla_hr = kla_s * 3600.0
    kla_err = np.sqrt(pcov[0, 0]) * 3600.0
    err_pct = abs(kla_hr - TARGET_KLA_HR) / TARGET_KLA_HR * 100.0

    print(f'Fitted  k_La = {kla_hr:.2f} +/- {kla_err:.2f} hr-1')
    print(f'Target  k_La = {TARGET_KLA_HR:.1f} hr-1')
    print(f'Error        = {err_pct:.1f} %')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    t_fit = np.linspace(0, t[-1], 500)

    axes[0].plot(t, C_f, 'o', ms=2, alpha=0.6, label='Simulation (integrated rate)')
    axes[0].plot(t_fit, saturation_model(t_fit, kla_s), 'r-', lw=2,
                 label=f'Fit: k_La = {kla_hr:.2f} hr-1')
    axes[0].axhline(C_sat, color='k', ls=':', label=f'C* = {C_sat:.3f} mol/m3')
    axes[0].set(xlabel='Physical time [s]', ylabel='C_f  [mol/m3]',
                title='Bulk dissolved O2 -- saturation curve fit')
    axes[0].legend()

    # Linearised: y = -ln(1 - C_f/C*)  vs t,  slope = k_La
    safe = C_f < 0.999 * C_sat
    y_lin = -np.log(1.0 - C_f[safe] / C_sat)
    axes[1].plot(t[safe], y_lin, 'o', ms=2, alpha=0.6, label='Simulation')
    axes[1].plot(t[safe], kla_s * t[safe], 'r-', lw=2,
                 label=f'k_La = {kla_hr:.2f} hr-1 (slope)')
    axes[1].set(xlabel='Physical time [s]', ylabel='-ln(1 - C_f/C*)',
                title='Linearised saturation plot')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(RUN_DIR / 'kla_fit.png', bbox_inches='tight')
    plt.show()

    status = 'PASS' if err_pct < 20.0 else 'FAIL'
    print(f'\n{"="*52}')
    print(f'  k_La  {kla_hr:.2f} hr-1  vs target {TARGET_KLA_HR} hr-1  ->  {status}')
    print(f'{"="*52}')
else:
    print('Skipping k_La fit -- C_f data not available or simulation not run yet.')

## 5  Energy balance: shaft power vs total dissipation

In steady state P_shaft = tau_z * omega = total dissipation.  
Acceptance criterion: |P_shaft - P_diss| / P_shaft < 5 %.

thermodynamics.txt column layout: `step  time[s]  KE  TE  ...`  
Column 1 is already physical time in seconds (AMReX default) -- do not
multiply by dt_phys.

In [ ]:
thermo_file = RUN_DIR / 'thermodynamics.txt'
if thermo_file.exists():
    tdf = pd.read_csv(thermo_file, comment='#', sep=r'\s+', header=None)
    print('thermodynamics.txt shape:', tdf.shape)
    print(tdf.head())

    if tdf.shape[1] >= 3:
        # Column 1 = physical time [s] (AMReX writes it in SI directly).
        # Do NOT multiply by dt_phys -- that would give s^2.
        t_thermo = tdf.iloc[:, 1].values     # s
        KE       = tdf.iloc[:, 2].values     # kinetic energy [LB units]

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(t_thermo, KE, label='Kinetic energy (LB)')
        ax.set(xlabel='Physical time [s]', ylabel='Kinetic energy [LB]',
               title='Kinetic energy vs time')
        ax.legend()
        plt.tight_layout()
        plt.savefig(RUN_DIR / 'kinetic_energy.png', bbox_inches='tight')
        plt.show()

        KE_tail = KE[int(0.8 * len(KE)):]
        rel_std = KE_tail.std() / KE_tail.mean() * 100.0
        print(f'KE relative std (last 20%): {rel_std:.1f} %  '
              f'({"steady" if rel_std < 5.0 else "not yet steady"})')
    else:
        print('Unexpected column count in thermodynamics.txt')
else:
    print(f'thermodynamics.txt not found in {RUN_DIR}.')

In [ ]:
# Shaft power from forces.txt: P_shaft = Tz_LB * omega_LB  [LB energy/step]
OMEGA_LB   = 0.002   # rad/step (body.angular_velocity in .inp)

forces_file = RUN_DIR / 'forces.txt'
if forces_file.exists():
    fdf = pd.read_csv(forces_file, comment='#', sep=r'\s+', header=None)
    print('forces.txt shape:', fdf.shape)
    print(fdf.head())

    if fdf.shape[1] >= 8:
        t_f  = fdf.iloc[:, 1].values   # s (physical time, SI)
        Tz   = fdf.iloc[:, 7].values   # torque about z [LB units]
        P_sh = Tz * OMEGA_LB           # P_shaft [LB energy/step]

        window = max(1, len(Tz) // 20)
        Tz_sm  = pd.Series(Tz).rolling(window, center=True, min_periods=1).mean()
        P_sm   = pd.Series(P_sh).rolling(window, center=True, min_periods=1).mean()

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].plot(t_f, Tz,    lw=0.8, alpha=0.5, label='raw')
        axes[0].plot(t_f, Tz_sm, lw=2,              label=f'Rolling mean w={window}')
        axes[0].set(xlabel='Physical time [s]', ylabel='Tz [LB]', title='Impeller torque')
        axes[0].legend()

        axes[1].plot(t_f, P_sh, lw=0.8, alpha=0.5, label='raw')
        axes[1].plot(t_f, P_sm, lw=2,               label='smoothed')
        axes[1].set(xlabel='Physical time [s]', ylabel='P_shaft [LB energy/step]',
                    title='Shaft power vs time')
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(RUN_DIR / 'shaft_power.png', bbox_inches='tight')
        plt.show()

        i_qs = int(0.8 * len(t_f))
        print(f'Quasi-steady Tz     = {Tz[i_qs:].mean():.4f} LB')
        print(f'Quasi-steady P_sh   = {P_sh[i_qs:].mean():.6f} LB energy/step')
        print('Verify P_shaft ~= P_diss by checking KE plateau std < 5% in sec 5a.')
    else:
        print('Unexpected column count in forces.txt')
else:
    print(f'forces.txt not found in {RUN_DIR}.')

## 6  Spatial O2 distribution (optional -- requires yt)

In [ ]:
import importlib
HAS_YT = importlib.util.find_spec('yt') is not None
plt_dirs = sorted(RUN_DIR.glob('plt?????'))
last_plt = plt_dirs[-1] if plt_dirs else None
print(f'yt available: {HAS_YT}')
print(f'Last plotfile: {last_plt}' if last_plt else 'No plt????? dirs found.')

In [ ]:
if HAS_YT and last_plt:
    import yt
    ds = yt.load(str(last_plt))
    comp0 = next((f for f in ds.field_list
                  if 'component_0' in f[1].lower() or 'scalar_0' in f[1].lower()), None)
    if comp0:
        slc = ds.slice('x', 90.0 * dx_phys)
        frb = slc.to_frb((180.0 * dx_phys, 'm'), 180)
        C_mol = np.array(frb[comp0]) * C_ref
        fig, ax = plt.subplots(figsize=(5, 8))
        im = ax.imshow(C_mol, origin='lower',
                       extent=[0, T_m, 0, 180.0 * dx_phys],
                       cmap='viridis', vmin=0, vmax=C_sat)
        plt.colorbar(im, ax=ax, label='C_O2  [mol/m3]')
        ax.set(xlabel='y [m]', ylabel='z [m]',
               title=f'Dissolved O2 -- x midplane @ {last_plt.name}')
        plt.tight_layout()
        plt.savefig(RUN_DIR / 'O2_spatial_slice.png', bbox_inches='tight')
        plt.show()
    else:
        print('component_0 field not found.  Available:', ds.field_list)
elif not HAS_YT:
    print('Install yt:  pip install yt')
else:
    print('No plotfile found -- run the simulation first.')

## 7  Validation summary

In [ ]:
print('\n' + '='*58)
print('  VALIDATION SUMMARY')
print('='*58)
print(f"  {'Quantity':<38} {'Sim':>8}  {'Target':>7}  Status")
print('-'*58)

try:
    s = 'PASS' if abs(kla_hr - TARGET_KLA_HR)/TARGET_KLA_HR < 0.20 else 'FAIL'
    print(f"  {'k_La  [hr-1]':<38} {kla_hr:>8.2f}  {TARGET_KLA_HR:>7.1f}  {s}")
except NameError:
    print(f"  {'k_La  [hr-1]':<38} {'N/A':>8}  {TARGET_KLA_HR:>7.1f}  ----")

if d_col and d_col in bdf.columns:
    i_qs = int(0.8 * len(t))
    d_qs = bdf[d_col].values[i_qs:].mean()
    s = 'PASS' if abs(d_qs - 3.0)/3.0 < 0.30 else 'FAIL'
    print(f"  {'Mean bubble diam (last 20%) [mm]':<38} {d_qs:>8.2f}  {'3.0':>7}  {s}")
else:
    print(f"  {'Mean bubble diam [mm]':<38} {'N/A':>8}  {'3.0':>7}  ----")

print('='*58)
print('Tolerance: k_La +-20%, bubble diam +-30%.')
print('Energy balance: KE plateau std < 5% indicates steady state.')